# 08. Many series on Apple Silicon

Forecasting one series per store, sensor or product means hundreds of independent fits, and fitting them one at a time with CmdStan adds up. On a Mac with Apple Silicon, NeoProphet's experimental MLX backend fits many series in one batched optimisation on the GPU. This notebook shows when that pays off, how to use it, how close its forecasts are to CmdStan's and where it does not apply.

The MLX cells need Apple Silicon and the `neoprophet[mlx]` extra. Anywhere else they print a short note and skip, and the CmdStan cells still run.

- A batch of series and the one-by-one baseline
- Is MLX available here?
- One model: `stan_backend="MLX"`
- Many models: `fit_many`
- How the gap grows with the number of series
- How close are the forecasts?
- Regressors, holidays and seasonalities: `configure=`
- Limits

Previous: [07. Reports and dashboards](07_reports_and_dashboards.ipynb) | Next: [09. Benchmark against upstream](09_benchmark_vs_upstream.ipynb)

In [ ]:
import importlib.util
import platform
import statistics
import time
from collections.abc import Callable
from importlib.metadata import version
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from prophet import Prophet

from helpers import load_example, quiet_logs

quiet_logs()

## A batch of series and the one-by-one baseline

Each CmdStan fit is an independent optimisation run in its own process, so the cost of a portfolio grows in a straight line with the number of series. To measure that we need a batch. The cell below builds 50 variants of the Peyton Manning page-view series, each with its own seeded noise, drift and level shift, so no two series are identical.

In [ ]:
BASE = load_example("peyton_manning")
N_SERIES = 50
# Point forecasts are all this notebook compares. Uncertainty intervals would dominate the
# time of every predict call, so they are switched off for the batch (section 3 keeps them).
FIT_KWARGS = {"uncertainty_samples": 0}
# Timing charts use plain matplotlib; library forecast plots keep the library style.
TIMING_STYLE = {
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "figure.dpi": 110,
    "font.size": 10,
}
CMDSTAN_COLOUR, MLX_COLOUR = "#1b6ca8", "#8e44ad"


def noisy_variants(df: pd.DataFrame, n: int, seed: int = 7) -> list[pd.DataFrame]:
    """``n`` variants of one series: seeded noise, a random drift and a random level shift.

    Variant ``i`` uses its own generator (``seed + i``), so it is the same whatever ``n`` is.
    """
    sd = float(df["y"].std())
    ramp = np.linspace(0.0, 1.0, len(df))
    variants = []
    for i in range(n):
        rng = np.random.default_rng(seed + i)
        noise = rng.normal(0.0, 0.05 * sd, len(df))
        drift = rng.normal(0.0, 0.3 * sd) * ramp
        shift = rng.normal(0.0, 0.2 * sd)
        variants.append(df.assign(y=df["y"] + noise + drift + shift))
    return variants


def timed[T](fn: Callable[..., T], *args: Any, **kwargs: Any) -> tuple[T, float]:
    """Call ``fn`` and return its result with the wall time in seconds."""
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, time.perf_counter() - start


def fit_cmdstan(
    df: pd.DataFrame, configure: Callable[[Prophet], None] | None = None, **kwargs: Any
) -> Prophet:
    """Fit one Prophet model with the default CmdStan backend."""
    model = Prophet(**kwargs)
    if configure is not None:
        configure(model)
    return model.fit(df)


batch = noisy_variants(BASE, N_SERIES)

recent = BASE["ds"] >= BASE["ds"].iloc[-1] - pd.Timedelta(days=500)
with plt.rc_context(TIMING_STYLE):
    fig, ax = plt.subplots(figsize=(10, 3.6))
    for variant in batch:
        ax.plot(variant.loc[recent, "ds"], variant.loc[recent, "y"], color="#999999", lw=0.5, alpha=0.5)
    ax.plot(BASE.loc[recent, "ds"], BASE.loc[recent, "y"], color="#222222", lw=1.2, label="original series")
    ax.set_ylabel("y (log page views)")
    ax.set_title(f"{N_SERIES} noisy variants (grey) around the original series, last 500 days")
    ax.legend(frameon=False)
    plt.show()

The baseline is the plain loop: one `Prophet` per series, fitted one after another. One untimed fit comes first, because the first CmdStan call in a process loads the compiled model. Spreading the fits over several CPU cores, as `tune` does, would narrow the gap shown below. That is not measured here.

In [ ]:
fit_cmdstan(batch[0], **FIT_KWARGS)  # untimed: loads the CmdStan model once

cmdstan_models: list[Prophet] = []
cmdstan_secs: list[float] = []
for frame in batch:
    model, secs = timed(fit_cmdstan, frame, **FIT_KWARGS)
    cmdstan_models.append(model)
    cmdstan_secs.append(secs)

mean_fit = statistics.fmean(cmdstan_secs)
print(f"CmdStan, one by one: {sum(cmdstan_secs):.1f} s for {N_SERIES} series ({mean_fit:.2f} s each)")
for n in (1_000, 10_000):
    print(f"At that rate {n:,} series of this length take about {n * mean_fit / 60:.0f} minutes.")

## Is MLX available here?

MLX is Apple's array framework for Apple Silicon. NeoProphet imports it lazily, so `import prophet` works without it, but the MLX backend needs a Mac with an M-series chip and the `neoprophet[mlx]` extra. The check below runs the same tests in order and says which one failed. Every MLX cell that follows is guarded by its result, so on any other machine they print a note and the notebook still runs to the end.

In [ ]:
def mlx_status() -> tuple[bool, str]:
    """Whether the MLX backend can run here, with a one-line reason either way."""
    if not (platform.system() == "Darwin" and platform.machine() == "arm64"):
        return False, f"MLX needs Apple Silicon; this machine is {platform.system()} {platform.machine()}."
    if importlib.util.find_spec("mlx") is None:
        return False, "MLX is not installed. Add it with: uv add 'neoprophet[mlx]'"
    try:
        import mlx.core as mx
    except ImportError as err:
        return False, f"MLX is installed but cannot be imported: {err}"
    if not mx.metal.is_available():
        return False, "MLX is installed but reports no Metal GPU."
    return True, f"MLX {version('mlx')}, default device {mx.default_device()}"


HAVE_MLX, MLX_NOTE = mlx_status()
SKIPPED = "Skipped: the MLX backend is not available here (see 'Is MLX available here?')."
mlx_models: list[Prophet] | None = None  # filled in below when MLX is available

print(MLX_NOTE)
if not HAVE_MLX:
    print("The MLX cells below will print a note and skip. The CmdStan cells still run.")

## One model: `stan_backend="MLX"`

For a single series, pass `stan_backend="MLX"` to `Prophet` and use it as usual. The model is the same, only the optimiser differs: MLX evaluates Prophet's MAP objective directly instead of calling Stan. This is the one place the notebook keeps the default uncertainty intervals, so the chart is an ordinary Prophet forecast.

In [ ]:
if not HAVE_MLX:
    print(SKIPPED)
else:
    m_cmdstan, t_cmdstan = timed(fit_cmdstan, BASE)
    _, t_first = timed(lambda: Prophet(stan_backend="MLX").fit(BASE))  # pays MLX start-up
    m_mlx, t_mlx = timed(lambda: Prophet(stan_backend="MLX").fit(BASE))

    future = m_mlx.make_future_dataframe(periods=365)
    fcst_mlx = m_mlx.predict(future)
    fcst_cmdstan = m_cmdstan.predict(future)

    gap = (fcst_mlx["yhat"] - fcst_cmdstan["yhat"]).abs().max() / (BASE["y"].max() - BASE["y"].min())
    print(f"Backend in use: {m_mlx.stan_backend.get_type()}")
    print(f"CmdStan fit {t_cmdstan:.2f} s | MLX first call {t_first:.2f} s (includes start-up) | MLX second call {t_mlx:.2f} s")
    print(f"Largest yhat difference to CmdStan over history and horizon: {gap:.2%} of the y range")
    fig = m_mlx.plot(fcst_mlx)
    plt.show()

## Many models: `fit_many`

`fit_many` takes a list of dataframes and returns a list of fitted models, in the same order, ready for `predict`. It builds each model, pads the series to a common size and optimises all of them together, so the GPU does the heavy arithmetic once for the whole batch. Keyword arguments go to every `Prophet(...)`. One untimed call comes first so one-off graph compilation is not charged to the timed runs, which are the median of three.

In [ ]:
if not HAVE_MLX:
    print(SKIPPED)
else:
    from prophet.mlx_backend import fit_many

    _, t_warm = timed(fit_many, batch[:2], **FIT_KWARGS)
    runs = [timed(fit_many, batch, **FIT_KWARGS) for _ in range(3)]
    mlx_models = runs[0][0]
    mlx_total = statistics.median(secs for _, secs in runs)

    print(f"Warm-up call (untimed in the comparison): {t_warm:.2f} s")
    print(f"fit_many on {N_SERIES} series: {mlx_total:.2f} s ({1000 * mlx_total / N_SERIES:.0f} ms per series)")
    print(f"CmdStan one by one:           {sum(cmdstan_secs):.2f} s ({1000 * mean_fit:.0f} ms per series)")
    print(f"Speed-up: {sum(cmdstan_secs) / mlx_total:.1f}x")

## How the gap grows with the number of series

CmdStan's total is the sum of its fits, so it grows in a straight line. A batched optimisation shares its set-up across the batch, so the cost per series should fall as the batch grows. The CmdStan line below uses the measured fits for up to 50 series and extends at the mean time per fit beyond that, shown dashed, to keep the notebook quick.

In [ ]:
SIZES = (1, 5, 10, 25, 50, 100, 200)
cumulative = np.cumsum(cmdstan_secs)


def cmdstan_seconds(n: int) -> float:
    """Measured time for the first ``n`` fits, or an extrapolation at the mean time per fit."""
    return float(cumulative[n - 1]) if n <= len(cumulative) else mean_fit * n


def mlx_seconds(n: int, repeats: int = 3) -> float:
    """Median wall time of ``fit_many`` on ``n`` series."""
    frames = batch[:n] if n <= len(batch) else noisy_variants(BASE, n)
    return statistics.median(timed(fit_many, frames, **FIT_KWARGS)[1] for _ in range(repeats))


if not HAVE_MLX:
    print(SKIPPED)
else:
    scaling = pd.DataFrame(
        {
            "CmdStan one by one (s)": [cmdstan_seconds(n) for n in SIZES],
            "CmdStan measured": [n <= len(batch) for n in SIZES],
            "fit_many (s)": [mlx_seconds(n) for n in SIZES],
        },
        index=pd.Index(SIZES, name="series"),
    )
    scaling["speed-up"] = scaling["CmdStan one by one (s)"] / scaling["fit_many (s)"]
    scaling["fit_many ms per series"] = 1000 * scaling["fit_many (s)"] / scaling.index
    display(scaling.round(3))

    measured = scaling[scaling["CmdStan measured"]]
    extended = scaling.loc[measured.index[-1] :]
    with plt.rc_context(TIMING_STYLE):
        fig, ax = plt.subplots(figsize=(7.5, 4))
        ax.plot(measured.index, measured["CmdStan one by one (s)"], "o-", color=CMDSTAN_COLOUR, label="CmdStan, one by one")
        ax.plot(extended.index, extended["CmdStan one by one (s)"], "o--", color=CMDSTAN_COLOUR, alpha=0.6)
        ax.plot(scaling.index, scaling["fit_many (s)"], "o-", color=MLX_COLOUR, label="MLX fit_many")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xticks(SIZES, [str(n) for n in SIZES])
        ax.set_xlabel("number of series")
        ax.set_ylabel("time to fit all series (s, log scale)")
        ax.set_title("Dashed: CmdStan extrapolated beyond the measured fits")
        ax.legend(frameon=False)
        fig.tight_layout()
        plt.show()

    first, last = scaling.iloc[0], scaling.iloc[-1]
    print(
        f"fit_many costs {first['fit_many ms per series']:.0f} ms per series at {SIZES[0]} series and "
        f"{last['fit_many ms per series']:.0f} ms at {SIZES[-1]}; CmdStan stays near {1000 * mean_fit:.0f} ms. "
        f"Speed-up: {first['speed-up']:.1f}x at {SIZES[0]} series, {last['speed-up']:.1f}x at {SIZES[-1]}."
    )

## How close are the forecasts?

Speed is only useful if the forecasts agree. Both backends minimise the same objective, but the optimisers, the arithmetic precision (float32 on the GPU) and their stopping rules differ. For each of the 50 series the cell below compares the two point forecasts over the history and 90 days beyond it, as the largest absolute difference divided by that series' own range. It also compares the objective each backend reached, which is minus Stan's `lp__`.

In [ ]:
HORIZON = 90


def yhat_with_horizon(model: Prophet, horizon: int) -> np.ndarray:
    """Point forecast over the history plus ``horizon`` days (no regressors)."""
    return model.predict(model.make_future_dataframe(periods=horizon))["yhat"].to_numpy()


def relative_gap(a: np.ndarray, b: np.ndarray, y: pd.Series) -> float:
    """Largest absolute difference between two forecasts as a share of the range of ``y``."""
    return float(np.max(np.abs(a - b)) / (y.max() - y.min()))


def objective(model: Prophet) -> float:
    """Negative log posterior the optimiser reached (minus Stan's ``lp__``)."""
    return -float(model.params["lp__"][0, 0])


if not HAVE_MLX:
    print(SKIPPED)
else:
    yhat_cmdstan = [yhat_with_horizon(m, HORIZON) for m in cmdstan_models]
    yhat_mlx = [yhat_with_horizon(m, HORIZON) for m in mlx_models]
    gaps = np.array([relative_gap(a, b, f["y"]) for a, b, f in zip(yhat_mlx, yhat_cmdstan, batch, strict=True)])
    better = np.array([objective(a) - objective(b) for a, b in zip(mlx_models, cmdstan_models, strict=True)])
    worst = int(np.argmax(gaps))

    with plt.rc_context(TIMING_STYLE):
        fig, (ax_hist, ax_worst, ax_diff) = plt.subplots(
            1, 3, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1, 1.3, 1.3]}
        )
        ax_hist.hist(100 * gaps, bins=15, color=MLX_COLOUR, alpha=0.8)
        ax_hist.set_xlabel("largest forecast difference (% of the series range)")
        ax_hist.set_ylabel("number of series")
        ax_hist.set_title("Gap to CmdStan, all 50 series")

        ds = mlx_models[worst].make_future_dataframe(periods=HORIZON)["ds"]
        tail = slice(-(365 + HORIZON), None)
        ax_worst.plot(ds.iloc[tail], yhat_cmdstan[worst][tail], color=CMDSTAN_COLOUR, lw=2, label="CmdStan")
        ax_worst.plot(ds.iloc[tail], yhat_mlx[worst][tail], color=MLX_COLOUR, lw=1.2, ls="--", label="MLX")
        ax_worst.set_title(f"Worst series (#{worst}), last year plus horizon")
        ax_worst.tick_params(axis="x", rotation=30)
        ax_worst.legend(frameon=False)

        ax_diff.plot(ds.iloc[tail], yhat_mlx[worst][tail] - yhat_cmdstan[worst][tail], color="#555555", lw=1)
        ax_diff.axhline(0, color="#999999", lw=0.8)
        ax_diff.set_title("MLX minus CmdStan, same series")
        ax_diff.tick_params(axis="x", rotation=30)
        fig.tight_layout()
        plt.show()

    print(f"Forecast gap to CmdStan: median {100 * np.median(gaps):.2f}%, worst {100 * gaps.max():.2f}% of the series range.")
    print(
        f"Objective reached: MLX lower or equal (within 0.01 nats) on {int((better <= 0.01).sum())} of {N_SERIES} series; "
        f"median difference {np.median(better):+.2f} nats (negative means MLX found the better optimum)."
    )

The two forecasts agree to a small fraction of the series range. The objective check explains most of the remaining gap. Stan's L-BFGS stops a little short of the optimum at the sharp kinks the changepoint prior creates, while the MLX solver usually gets slightly closer, so the two are not expected to match exactly and neither is "the" right answer.

## Regressors, holidays and seasonalities: `configure=`

A plain `Prophet(...)` call cannot add regressors or custom seasonalities, because that happens after construction. `fit_many` therefore takes a `configure` function that it calls on every fresh model before preprocessing, so you can call `add_regressor`, `add_seasonality`, `add_country_holidays` and friends there. The hook must not fit the model. Regressor columns go in the dataframes as usual.

Here every series gets US holidays, a monthly seasonality and a stylised `nfl_season` regressor (one from September to January). The hook is shared, so every model is configured identically.

In [ ]:
def nfl_season(ds: pd.Series) -> pd.Series:
    """1 from September to January (a stylised season), 0 otherwise. Known for future dates."""
    month = ds.dt.month
    return ((month >= 9) | (month == 1)).astype(float)


def configure(model: Prophet) -> None:
    model.add_country_holidays("US")
    model.add_seasonality("monthly", period=30.5, fourier_order=5)
    model.add_regressor("nfl_season")


N_CONFIG = 10
config_frames = [f.assign(nfl_season=nfl_season(f["ds"])) for f in batch[:N_CONFIG]]

config_cmdstan, t_config_cmdstan = timed(lambda: [fit_cmdstan(f, configure, **FIT_KWARGS) for f in config_frames])
print(f"CmdStan, one by one, {N_CONFIG} configured series: {t_config_cmdstan:.1f} s")

if not HAVE_MLX:
    print(SKIPPED)
else:
    config_mlx, t_config_mlx = timed(fit_many, config_frames, configure=configure, **FIT_KWARGS)
    print(f"fit_many with configure=:                          {t_config_mlx:.2f} s")
    print(f"Regressors in the first MLX model: {list(config_mlx[0].extra_regressors)}")

    model = config_mlx[0]
    future = model.make_future_dataframe(periods=365)
    future["nfl_season"] = nfl_season(future["ds"])  # regressors must be known for the horizon
    fig = model.plot_components(model.predict(future))
    plt.show()

Does the fidelity hold beyond the default setup? The table below fits small batches of series under four different configurations with both backends and compares them in-sample. The first row reuses the hook above. The next two use `Prophet` keyword arguments, which `fit_many` passes through (`seasonality_mode="multiplicative"` and `growth="flat"`), on the same ten series. The last row fits the two hourly series of `pedestrians_multivariate` as separate targets with `yearly_seasonality=False`. The gaps are the largest in-sample forecast difference as a percentage of each series' range.

In [ ]:
def yhat_in_sample(model: Prophet, frame: pd.DataFrame) -> np.ndarray:
    """Point forecast for the rows of ``frame`` (regressor columns included, ``y`` dropped)."""
    return model.predict(frame.drop(columns="y"))["yhat"].to_numpy()


pedestrians = load_example("pedestrians_multivariate")
hourly_frames = [pedestrians[["ds", col]].rename(columns={col: "y"}) for col in ("location_4", "location_41")]

CASES: dict[str, tuple[list[pd.DataFrame], Callable[[Prophet], None] | None, dict[str, Any]]] = {
    "configure= hook": (config_frames, configure, {}),
    "multiplicative seasonality": (batch[:N_CONFIG], None, {"seasonality_mode": "multiplicative"}),
    "flat growth": (batch[:N_CONFIG], None, {"growth": "flat"}),
    "hourly pedestrians": (hourly_frames, None, {"yearly_seasonality": False}),
}

if not HAVE_MLX:
    print(SKIPPED)
else:
    rows = {}
    for label, (frames, hook, kwargs) in CASES.items():
        kwargs = {**FIT_KWARGS, **kwargs}
        cs_models, t_cs = timed(lambda: [fit_cmdstan(f, hook, **kwargs) for f in frames])
        ml_models, t_ml = timed(fit_many, frames, configure=hook, **kwargs)
        diffs = [
            relative_gap(yhat_in_sample(a, f), yhat_in_sample(b, f), f["y"])
            for a, b, f in zip(ml_models, cs_models, frames, strict=True)
        ]
        rows[label] = {
            "series": len(frames),
            "CmdStan (s)": t_cs,
            "fit_many (s)": t_ml,
            "median gap (%)": 100 * float(np.median(diffs)),
            "worst gap (%)": 100 * max(diffs),
        }
    display(pd.DataFrame.from_dict(rows, orient="index").round(2))

## Limits

The MLX backend is experimental and deliberately narrow. It re-implements the MAP objective of `prophet.stan` rather than calling Stan, so only what that covers is supported. The next cells show each limit rather than just listing it.

### What it cannot fit

Logistic growth and MCMC sampling are not implemented. The errors are explicit, so a mistake is caught at the call.

In [ ]:
if not HAVE_MLX:
    print(SKIPPED)
else:
    for label, kwargs in {"logistic growth": {"growth": "logistic"}, "MCMC sampling": {"mcmc_samples": 50}}.items():
        try:
            fit_many(batch[:2], **kwargs)
        except NotImplementedError as err:
            print(f"{label}: {err}")

### Precision

The GPU has no float64, so `device="gpu"` (the default) optimises in float32. `device="cpu"` runs the same solver in float64 and is meant for parity checks. Setting the `NEOPROPHET_MLX_DEVICE` environment variable changes the default. The cell compares both devices on ten series. If float32 were the main source of the gap to CmdStan, the CPU run would sit much closer to CmdStan than the GPU run does.

In [ ]:
if not HAVE_MLX:
    print(SKIPPED)
else:
    n_check = 10
    by_device = {dev: fit_many(batch[:n_check], device=dev, **FIT_KWARGS) for dev in ("gpu", "cpu")}
    yhat_dev = {dev: [yhat_in_sample(m, f) for m, f in zip(ms, batch)] for dev, ms in by_device.items()}
    yhat_ref = [yhat_in_sample(m, f) for m, f in zip(cmdstan_models, batch)]

    def worst_gap(a: list[np.ndarray], b: list[np.ndarray]) -> float:
        return 100 * max(relative_gap(x, z, f["y"]) for x, z, f in zip(a, b, batch))

    print(f"Worst gap in-sample over {n_check} series, % of the series range:")
    print(f"  GPU float32 vs CmdStan: {worst_gap(yhat_dev['gpu'], yhat_ref):.2f}")
    print(f"  CPU float64 vs CmdStan: {worst_gap(yhat_dev['cpu'], yhat_ref):.2f}")
    print(f"  GPU float32 vs CPU float64: {worst_gap(yhat_dev['gpu'], yhat_dev['cpu']):.2f}")
    closed = 1 - worst_gap(yhat_dev["cpu"], yhat_ref) / worst_gap(yhat_dev["gpu"], yhat_ref)
    print(f"Float64 on the CPU closes {closed:.0%} of the worst-case gap to CmdStan.")

### Short series and mixed lengths

The GPU needs enough work to pay for its fixed costs. For one tiny series a CmdStan fit finishes in a few hundredths of a second and MLX can be slower. The table times a single fit and a batch of 50 for three series lengths (single fits are the median of five). `fit_many` also pads every series in a call to the longest one, so the last cell checks what one long series does to a batch of short ones. For very uneven collections, group series of similar length into separate calls.

In [ ]:
SHORT_CASES = {
    "Air Passengers": ("air_passengers", {"seasonality_mode": "multiplicative"}),
    "Retail Sales": ("retail_sales", {}),
    "Peyton Manning": ("peyton_manning", {}),
}


def median_seconds(fn: Callable[[], object], repeats: int = 5) -> float:
    return statistics.median(timed(fn)[1] for _ in range(repeats))


if not HAVE_MLX:
    print(SKIPPED)
else:
    rows = {}
    for label, (name, kwargs) in SHORT_CASES.items():
        series = load_example(name)
        kwargs = {**FIT_KWARGS, **kwargs}
        frames = noisy_variants(series, N_SERIES)
        fit_cmdstan(frames[0], **kwargs)  # untimed warm-up
        fit_many(frames[:2], **kwargs)
        rows[label] = {
            "rows": len(series),
            "CmdStan, 1 (ms)": 1000 * median_seconds(lambda: fit_cmdstan(frames[0], **kwargs)),
            "MLX, 1 (ms)": 1000 * median_seconds(lambda: Prophet(stan_backend="MLX", **kwargs).fit(frames[0])),
            f"CmdStan, {N_SERIES} (s)": timed(lambda: [fit_cmdstan(f, **kwargs) for f in frames])[1],
            f"fit_many, {N_SERIES} (s)": timed(fit_many, frames, **kwargs)[1],
        }
    short = pd.DataFrame.from_dict(rows, orient="index")
    short["batch speed-up"] = short[f"CmdStan, {N_SERIES} (s)"] / short[f"fit_many, {N_SERIES} (s)"]
    display(short.round(2))

    retail = noisy_variants(load_example("retail_sales"), N_SERIES)
    mixed = [*retail[:-1], noisy_variants(BASE, 1)[0]]
    t_even = median_seconds(lambda: fit_many(retail, **FIT_KWARGS), repeats=3)
    t_mixed = median_seconds(lambda: fit_many(mixed, **FIT_KWARGS), repeats=3)
    print(
        f"{N_SERIES} Retail Sales series ({len(retail[0])} rows): {t_even:.2f} s. "
        f"The same batch with one Peyton Manning series ({len(BASE)} rows) swapped in: {t_mixed:.2f} s."
    )

### The rest

- Linear and flat growth only. Logistic growth needs the CmdStan backend.
- MAP estimation only. `mcmc_samples` above zero is rejected, so there are no posterior samples. Uncertainty intervals still work, because `predict` simulates them from the fitted trend as it does for any MAP fit.
- Experimental: the objective is a re-implementation of `prophet.stan`, checked against CmdStan in the test suite, but it is a second code path to trust.
- `fit_many` returns fitted models that predict like any other, but it does not report convergence per model. If some series miss the tolerance it logs a warning on the `prophet.mlx_backend` logger, so keep that logger at WARNING or below.
- Memory is bounded by `chunk_elements`: large batches are split into chunks automatically.

## Next

Use `fit_many` when you have dozens or more series of reasonable length and a Mac to run them on, and stay with CmdStan for single or tiny series, logistic growth or MCMC. The next notebook, [09. Benchmark against upstream](09_benchmark_vs_upstream.ipynb), measures NeoProphet against upstream Prophet in an isolated environment, including this many-series scenario at 10 and 100 series.

Previous: [07. Reports and dashboards](07_reports_and_dashboards.ipynb) | Next: [09. Benchmark against upstream](09_benchmark_vs_upstream.ipynb)